# 📈 04 — Evaluasi & Visualisasi
> Notebook ini mengevaluasi model secara menyeluruh: perbandingan metrik,
> prediksi vs aktual, feature importance, dan kesimpulan.

## 📦 Import Library

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from src.data_loader  import load_config, load_processed_train
from src.train        import get_models, split_data, train_all_models, load_model
from src.evaluate     import (
    evaluate_all, plot_model_comparison,
    plot_pred_vs_actual, plot_feature_importance
)

print('✅ Library berhasil diimpor.')

## 📂 Muat Data & Model

In [ ]:
cfg = load_config('../config.yaml')
df  = load_processed_train(f"../{cfg['data']['train_processed']}")

target = cfg['data']['target_column']
X = df.drop(columns=[target])
y = df[target]

X_train, X_valid, y_train, y_valid = split_data(
    X, y,
    test_size    = cfg['preprocessing']['test_size'],
    random_state = cfg['preprocessing']['random_state'],
)

In [ ]:
# Re-train semua model (atau load dari disk)
models = get_models(cfg['models'])
trained_models, preds = train_all_models(models, X_train, y_train, X_valid)

# Load model Random Forest yang disimpan
rf_model = load_model(f"../{cfg['output']['model_path']}")

## 📋 7. Ringkasan Performa Model

In [ ]:
results_df = evaluate_all(preds, y_valid)
print('\n📋 Tabel Performa:')
results_df

## 📊 7.1 Bar Chart Perbandingan Metrik

In [ ]:
plot_model_comparison(
    results_df,
    save_path=f"../{cfg['output']['figures_dir']}model_comparison.png"
)

## 🔵 7.2 Scatter Plot: Prediksi vs Aktual

In [ ]:
plot_pred_vs_actual(
    y_valid, preds,
    save_path=f"../{cfg['output']['figures_dir']}pred_vs_actual.png"
)

## 🌳 7.3 Feature Importance (Random Forest)

In [ ]:
plot_feature_importance(
    rf_model,
    feature_names = X.columns.tolist(),
    top_n         = 15,
    save_path     = f"../{cfg['output']['figures_dir']}feature_importance.png"
)

## ✅ 8. Kesimpulan
### Ringkasan Proyek
Pipeline prediksi harga rumah end-to-end berhasil dibangun.

### Temuan Utama
| Aspek | Temuan |
|---|---|
| Missing Values | `SalePrice` (test), `MSZoning`, `Exterior1st`, `BsmtFinSF2`, `TotalBsmtSF` — ditangani median/modus |
| Korelasi tertinggi | `TotalBsmtSF` dan `YearBuilt` terhadap `SalePrice` |
| Model terbaik | **Random Forest** — MAPE & R² terbaik |

### Rekomendasi Pengembangan
| Aspek | Rekomendasi |
|---|---|
| Feature Engineering | Tambah `AgeOfHouse = YearSold - YearBuilt` |
| Hyperparameter Tuning | Gunakan `GridSearchCV` atau `Optuna` |
| Model Lanjutan | Coba `XGBoost` / `LightGBM` |
| Validasi | Terapkan K-Fold Cross Validation |

In [ ]:
best = results_df['R²'].idxmax()
print('='*45)
print('  RINGKASAN HASIL EVALUASI')
print('='*45)
print(results_df.to_string())
print(f'\n🏆 Model Terbaik : {best}')
print(f'   R²            : {results_df.loc[best, "R²"]:.4f}')
print(f'   MAPE          : {results_df.loc[best, "MAPE (%)"]:.2f}%')
print(f'   MAE           : ${results_df.loc[best, "MAE ($)"]:,.0f}')